# Classic SPL v9.4.0 - loss ratio refit validation (cat excluded)Derived from `ngraf/spl/loss_ratio_implementation.ipynb`, repointed at`classic_spl_ltv` and restructured around the questions that are still open.## VERIFY BEFORE RUNNINGCells marked `# >>> VERIFY` contain names I could not confirm from what you haveshown me. Fix them first; a wrong path here produces a clean run against thewrong data, which is worse than a crash.Open items carried into this notebook:1. **`nt0` branch shape.** Nick's notebook treats NT0 as a single flat value   (`nt0_lr`). Production has `nt0_slope` + `nt0_int`. For line 16 both agree   (`nt0_slope=0`, flat at 1.0057). For the mirrored lines the two conventions   differ at nt6=1 by `slope_yr * 0.5` — about 0.007 on line 32. `FLAT_NT0`   below toggles between them. Confirm which the scorer implements.   (Note: mirroring `nt0_slope=slope_yr` is safe under *either* convention,   since it reproduces the renewal curve.)2. **Nick's notebook fits disagree with production** on lines 78 and 88, and it   carries commented-out variants. It is a different vintage. `PROD_FITS` here   is transcribed from `1_policy.py`, not from his notebook.3. **Cat add-back** is still unresolved. Section C is the test for it.

In [ ]:
# Dynaconf env vars must be set BEFORE the first import. Restart the kernel# when switching environments.# >>> VERIFY branch name.%env ENV_FOR_DYNACONF = prod%env DYNACONF_GIT_BRANCH = feature/B-2895875%env DYNACONF_GIT_CHECKOUT = feature/B-2895875

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as plt# >>> VERIFY these import paths against classic_spl_ltv.import ltv_helpers.non_spark_helpers as nsfrom classic_spl_ltv.config.paths import paths as p%matplotlib inlinepd.options.display.max_rows = 2000pd.options.display.max_columns = 500

In [ ]:
# >>> VERIFY every path. p.* is preferred over hand-assembled strings -# duplicate keys in paths.toml silently resolve to the first definition.P_SCORED_NEW = p.score_internal_resultsP_SCORED_UNBAL = p.score_internal_results_unbalanced# Prior release output - must NOT be on your branch.P_SCORED_PRIOR = "tmx-smsiweb/.../v9.3.1/score/score_internal_results/"  # >>> VERIFYprint(P_SCORED_NEW)print(P_SCORED_UNBAL)

## Section A - fits

In [ ]:
# Cat-excluded refit. slope_yr / intercept at 10 dp from the notebook's# other_lr_tuple print; nt0_int at 6 dp from spl_fits (only precision available).# >>> VERIFY precision convention before merging to 1_policy.py.NEW_FITS = {    # line: (slope_yr,       intercept,      nt0_slope,      nt0_int)    "16": (-0.0341583943,  0.6105496031,  0.0,            0.970097),    "32": (-0.0133350880,  0.3664844453, -0.0133350880,   0.3664844453),    "71": (-0.0435713887,  0.6252041479, -0.0435713887,   0.6252041479),    "72": ( 0.0,           0.3154266994,  0.0,            0.3154266994),    "78": ( 0.0,           0.4930239249,  0.0,            0.4930239249),    "88": ( 0.0,           0.7888171791,  0.0,            1.180678),    "90": (-0.0237481671,  0.5650788252,  0.0,            0.769902),}# Shipped fits, transcribed from classic_spl_ltv/jobs/score/1_policy.pyPROD_FITS = {    "16": (-0.02294,   0.6347,   0.0,       1.0057),    "32": (-0.01407,   0.6206,  -0.01407,   0.6206),    "71": (-0.05045,   0.7259,  -0.05045,   0.7259),    "72": (-0.003477,  0.6060,  -0.003477,  0.6060),    "78": (-0.03717,   0.82678, -0.03717,   0.82678),    "88": ( 0.0,       0.7571,   0.0,       1.2079),    "90": (-0.02329,   0.7527,   0.0,       0.9611),}LINE_NAMES = {"16": "Specialty Auto", "32": "Mfg Home", "71": "Renters",              "72": "Landlord", "78": "Condo", "88": "PUP", "90": "Boat"}

In [ ]:
# Fit evaluation, matching Nick's create_slope_intercept_spl_lr_dfFLAT_NT0 = True  # >>> VERIFY  True = Nick's flat nt0_lr; False = nt0_int + nt0_slope*ntrNT6_CAP = 18  # curve is capped at NTR = 9 (the censored bucket)def eval_fit(nt6, f):    """f = (slope_yr, intercept, nt0_slope, nt0_int). Returns loss ratio."""    slope_yr, intercept, nt0_slope, nt0_int = f    ntr = min(nt6, NT6_CAP) / 2    if nt6 < 2:        return nt0_int if FLAT_NT0 else nt0_int + nt0_slope * ntr    return intercept + slope_yr * ntrdef fit_table(fits, max_nt6=19):    rows = []    for line, f in sorted(fits.items()):        for nt6 in range(max_nt6 + 1):            rows.append(                {"drv_line": str(line), "nt6": nt6, "ntr": min(nt6, NT6_CAP) / 2,                 "lr_expected": eval_fit(nt6, f)}            )    return pd.DataFrame(rows)

### Read this before the pipeline sectionsThe NTR=0 column is the one to look at. Landlord and Condo moved to flat fits,and NT0 mirrors the renewal intercept, so:- Landlord NT0: 0.6060 -> 0.3154- Condo NT0:    0.8268 -> 0.4930Neither number was fit as a new-business estimate; both fell out of the flatfallback. Condo is the one to defend, since FL runs 4.34 at NTR=0.

In [ ]:
# Fit-level delta. No pipeline data needed - pure arithmetic on the two dicts.rows = []for line in sorted(NEW_FITS):    for nt6 in [0, 2, 4, 6, 10, 18]:        rows.append({            "drv_line": line, "name": LINE_NAMES[line], "ntr": min(nt6, NT6_CAP) / 2,            "prod": eval_fit(nt6, PROD_FITS[line]),            "new": eval_fit(nt6, NEW_FITS[line]),        })delta = pd.DataFrame(rows)delta["diff"] = delta["new"] - delta["prod"]delta.pivot_table(index=["drv_line", "name"], columns="ntr", values="diff").round(4)

## Section B - propagation check (weak, but necessary)Confirms the scorer picked up the new fits: every policy's `lr` should equalthe fit evaluated at its `nt6`.This proves the numbers reached the pipeline. It proves nothing about whetherthey are right - `loss = premium * lr` by construction, so dividing back iscircular. Same trap as the expense-ratio validation.

In [ ]:
df = ns.read_parquet_s3_to_pandas(P_SCORED_NEW)df["drv_line"] = df["drv_line"].astype(str)df = df[df["drv_line"].isin(NEW_FITS)]# >>> VERIFY: 'lr' is the applied loss ratio column; 'nt6' the tenure counter.df["lr_expected"] = [eval_fit(n, NEW_FITS[l])                     for n, l in zip(df["nt6"], df["drv_line"])]df["lr_gap"] = (df["lr"] - df["lr_expected"]).abs()chk = df.groupby("drv_line", as_index=False).agg(    n=("lr", "size"), max_gap=("lr_gap", "max"), mean_lr=("lr", "mean"))chk["pass"] = chk["max_gap"] < 1e-6chk.round(6)

## Section C - the cat add-back testThis is the section that matters.The scored output carries `loss` and `cat` as separate columns. So:- `lr_ex_cat  = loss / premium`- `lr_total   = (loss + cat) / premium`Run this on the new output and on the prior release, join, and diff.**If `lr_total` is roughly preserved across releases** while `lr_ex_cat` drops,the cat load is being added back downstream and open decision #1 answers itself.**If `lr_total` drops too**, cat has been removed from the P&L and nothing putit back - which means the -$784M is a real reserve gap, not a re-parameterisation.Then read `cat_load_preprocess` in `dodo_finance.py` either way, because if theprior cat load was calibrated alongside a cat-inclusive LR, reusing it under anex-cat LR is a mismatch in the opposite direction.

In [ ]:
# >>> VERIFY both paths. PRIOR must point at the previous release's output,# not at your branch.df_new = ns.read_parquet_s3_to_pandas(P_SCORED_NEW)df_old = ns.read_parquet_s3_to_pandas(P_SCORED_PRIOR)def lr_summary(df, tag):    d = df.copy()    d["drv_line"] = d["drv_line"].astype(str)    g = d.groupby("drv_line", as_index=False).agg(        premium=("premium", "sum"), loss=("loss", "sum"), cat=("cat", "sum")    )    g["lr_ex_cat"] = g["loss"] / g["premium"]    g["lr_total"] = (g["loss"] + g["cat"]) / g["premium"]    g["cat_share"] = g["cat"] / (g["loss"] + g["cat"])    g["src"] = tag    return gcmp = lr_summary(df_new, "new").merge(    lr_summary(df_old, "prior"), on="drv_line", suffixes=("_new", "_old"))cmp["d_ex_cat"] = cmp["lr_ex_cat_new"] - cmp["lr_ex_cat_old"]cmp["d_total"] = cmp["lr_total_new"] - cmp["lr_total_old"]cmp["d_premium"] = cmp["premium_new"] - cmp["premium_old"]cmp[["drv_line", "lr_ex_cat_old", "lr_ex_cat_new", "d_ex_cat",     "lr_total_old", "lr_total_new", "d_total", "d_premium"]].round(4)

### Line 88 is a free controlPUP's cat share is 0.0%, so cat removal should be a no-op for it. Any movementin line 88 between releases is **entirely** the data window (2023-2025 vs2023-2024), not the cat strip.Its renewal intercept went 0.7571 -> 0.7889 - **up**, while every other linefell hard. If the window effect is +0.03 on a clean line, part of your -15.3points is window, working against you. Quantify it before presenting.

In [ ]:
pup = cmp[cmp["drv_line"] == "88"]print("PUP cat share (new):  ", float(pup["cat_share_new"].iloc[0]))print("PUP cat share (prior):", float(pup["cat_share_old"].iloc[0]))print("PUP d_ex_cat (pure window effect):", float(pup["d_ex_cat"].iloc[0]))# Apply that window effect as an offset to the other lines to separate# window from cat. Crude, but it bounds the decomposition.w = float(pup["d_ex_cat"].iloc[0])cmp["d_ex_cat_less_window"] = cmp["d_ex_cat"] - wcmp[["drv_line", "d_ex_cat", "d_ex_cat_less_window"]].round(4)

### Dollar impact

In [ ]:
# Dollar impact, premium-weighted. This is the headline for the Loop page.# The unweighted mean over an NTR grid in Nick's In[26] is NOT comparable to a# scored average - do not use it.cmp["dollar_impact_ex_cat"] = cmp["d_ex_cat"] * cmp["premium_new"]tot_prem = cmp["premium_new"].sum()tot_impact = cmp["dollar_impact_ex_cat"].sum()out = cmp[["drv_line", "premium_new", "d_ex_cat", "dollar_impact_ex_cat"]].copy()out["pct_of_impact"] = out["dollar_impact_ex_cat"] / tot_impactprint(f"total premium: {tot_prem:,.0f}")print(f"total impact:  {tot_impact:,.0f}")print(f"points:        {tot_impact / tot_prem * 100:.1f}")out.sort_values("dollar_impact_ex_cat").round(4)

## Section D - Florida condoNot a keep/drop question. FL earns ~$794/exposure vs ~$414 non-FL whileex-cat loss per exposure is essentially equal ($222 vs $207) - so the low ratiois **pricing, not risk**, and FL's premium loading pays for the hurricanes thatwere just stripped from the numerator.Whether the carve-out survives depends entirely on how cat gets added back:- proportional to **premium** -> FL stays low, carve-out survives, and a pricing  artifact gets laundered into a risk parameter- proportional to **actual cat loss** -> FL likely inverts, carve-out dissolvesDo not run more FL cuts until that is decided. Until then this is unfalsifiable.

In [ ]:
fl = df_new[df_new["drv_line"].astype(str) == "78"].copy()fl["is_fl"] = fl["state"] == "FL"  # >>> VERIFY state column name / codeg = fl.groupby("is_fl", as_index=False).agg(    premium=("premium", "sum"), loss=("loss", "sum"), cat=("cat", "sum"))g["lr_ex_cat"] = g["loss"] / g["premium"]g["lr_total"] = (g["loss"] + g["cat"]) / g["premium"]g["cat_share"] = g["cat"] / (g["loss"] + g["cat"])g["prem_share"] = g["premium"] / g["premium"].sum()g.round(4)

## Section E - remaining verificationSmall, cheap, and all still open.

In [ ]:
# 1. Cat code '9' - is it a real PCS-style serial or an internal sentinel?#    A true catastrophe clusters in month x state. A sentinel does not.#    >>> Run this against the CLAIMS extract, not the scored output.## claims["CATCD"].value_counts(dropna=False)# pd.crosstab(claims.loc[claims["CATCD"] == "9", "ACTMO"],#             claims.loc[claims["CATCD"] == "9", "GEOST"])# 2. PUP zero cat - distinguish the two failure modes.#    Is CATCD populated-and-all-blank for line 88, or absent entirely?## claims.loc[claims["ALINE"] == "88", "CATCD"].isna().mean()# claims.loc[claims["ALINE"] == "88", "CATCD"].notna().sum()# 3. Exposure window control. Rerun the fits on 2023-2024 only.#    ACTYR 125 is immature; if losses are reported-to-date rather than developed,#    every level here is biased low and not comparable to the prior analysis.#    This is a control, not a decision - run it regardless.# 4. Landlord positive slope (+0.00733, p<0.001, $1.6B) suppressed by the#    'slope > 0 -> flat' rule. Refit with NTR=9 dropped:#      - positive only WITH the censored bucket -> rule is accidentally right#      - positive WITHOUT it -> the rule is suppressing real signal on the#        single largest line#    Cat removal strips the fattest tail from the residuals, so the post-cat#    p-value is MORE credible than any pre-cat one. That argues for the slope.